# Chapter 8: Building with A2A
## Multi-Agent Systems and Framework Integration - Code Examples

This notebook contains the runnable code from Chapter 8. It builds an A2A server with the official SDK, consumes it from a LangGraph agent, runs the chapter's workflow patterns against live agents, and secures a server with TLS, authentication, and role-based authorization:
- The chapter's weather executor and its production variant, served with `a2a-sdk` 1.1.2 and probed over JSON-RPC
- A LangChain tool that calls an A2A agent, driven by a `create_agent()` agent on `gpt-5.6-luna`
- The CrewAI, Google ADK, and Strands listings as reference cells, with the environment constraint that keeps them from running here
- The `delegate` helper, sequential delegation, parallel execution, and human-in-the-loop approval
- The healthcare referral workflow with its pre-authorization gate
- TLS 1.3 enforcement, bearer and API key middleware with a JWT validator, and the RBAC authorizer
- A model-driven orchestrator that plans a three-agent workflow over A2A

### Setup

The dependencies for every chapter are declared in `pyproject.toml` at the repository root. From the root, run:

```bash
uv sync --all-groups
```

Then start Jupyter with `uv run jupyter lab` and select the **Agentic AI Handbook (Python 3.13)** kernel.

Two parts call the OpenAI API. Copy `.env.example` to `.env` at the repository root and add your `OPENAI_API_KEY` before running the cells. The TLS part generates a self-signed certificate with the `openssl` command, which ships with macOS and most Linux distributions.

### How the servers run

Each A2A server is written to a file with a `%%writefile` cell and started as a background process on localhost. Part 1 starts the chapter's weather agent, Part 4 starts a scripted specialist agent that plays the data collector, analyzer, report writer, and hospital agents so the workflow outputs are stable, and the stop cell at the end ends both. The security part builds apps in the notebook process and tests them with an in-process HTTP client, and it starts one HTTPS server in a thread. Everything is written inside a temporary workspace created in the setup cell, so nothing in this repository is modified.

The chapter's listings use example domains such as `api.weather-services.com`. This notebook points them at the local agents instead. Every listing is otherwise reproduced as printed.

In [ ]:
# Import required libraries
import os
import sys
import json
import time
import uuid
import socket
import logging
import asyncio
import threading
import tempfile
import subprocess
from pathlib import Path
from dotenv import load_dotenv

import httpx
import requests

# Load environment variables (API keys)
load_dotenv()

# Verify API keys are loaded
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found in environment"

MODEL = "gpt-5.6-luna"

# Every server file lives in a throwaway workspace
WORKSPACE = Path(tempfile.mkdtemp(prefix="ch8-a2a-")).resolve()
os.chdir(WORKSPACE)

# Servers are launched with the same interpreter that runs this notebook
PYTHON = sys.executable


def wait_for_port(port: int, timeout: float = 10.0) -> None:
    """Block until a local port accepts connections."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            socket.create_connection(("127.0.0.1", port), timeout=0.2).close()
            return
        except OSError:
            time.sleep(0.2)
    raise RuntimeError(f"nothing is listening on port {port}")


print("Environment setup complete")
print(f"Workspace: {WORKSPACE}")

## Part 1: Building an A2A Server with the SDK

The chapter shows two executors and leaves the wiring to this notebook. The file below contains both executors exactly as printed, plus the pieces the chapter calls mechanical: the Agent Card, the SDK's `DefaultRequestHandler` with an in-memory task store, and the route factories that serve the card and the JSON-RPC binding on a Starlette app. `build_routes()` returns the routes on their own so the security part can put middleware in front of them.

`WeatherAgentExecutor` follows the 1.0 contract: the Task goes onto the event queue first, then an artifact and a terminal status. `ProductionExecutor` wraps the work in a guard and reports failures as `TASK_STATE_FAILED` rather than dropping the connection. The `_get_weather` stand-in returns a fixed forecast, and `_process` raises when the request contains the word fail, so both paths can be exercised.

In [ ]:
%%writefile weather_agent.py
"""Chapter 8's A2A server: the chapter's executors plus the wiring the chapter defers to this notebook."""
import logging
import sys

import uvicorn
from google.protobuf import json_format
from starlette.applications import Starlette

from a2a.helpers import (new_task_from_user_message, new_text_artifact_update_event,
                         new_text_status_update_event)
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import AgentCard, TaskState

logger = logging.getLogger("weather")


class WeatherAgentExecutor(AgentExecutor):
    async def execute(self,
                      context: RequestContext,
                      event_queue: EventQueue) -> None:
        forecast = await self._get_weather(context.get_user_input())

        # v1.0 contract: enqueue the Task first, then any update events
        task = context.current_task or \
               new_task_from_user_message(context.message)

        await event_queue.enqueue_event(task)
        await event_queue.enqueue_event(new_text_artifact_update_event(
            task_id=task.id,
            context_id=task.context_id,
            name="forecast",
            text=forecast))
        await event_queue.enqueue_event(new_text_status_update_event(
            task_id=task.id,
            context_id=task.context_id,
            state=TaskState.TASK_STATE_COMPLETED,
            text="Done"))

    async def cancel(self,
                     context: RequestContext,
                     event_queue: EventQueue) -> None:
        raise NotImplementedError("cancellation is not supported")

    async def _get_weather(self, location: str) -> str:
        return f"Sunny and 22 C in {location}"


class ProductionExecutor(WeatherAgentExecutor):
    async def execute(self,
                      context: RequestContext,
                      event_queue: EventQueue) -> None:
        task = context.current_task or \
               new_task_from_user_message(context.message)

        await event_queue.enqueue_event(task)
        try:
            result = await self._process(context.get_user_input())
            await event_queue.enqueue_event(new_text_artifact_update_event(
                task_id=task.id, context_id=task.context_id,
                name="result", text=result))
            await event_queue.enqueue_event(new_text_status_update_event(
                task_id=task.id, context_id=task.context_id,
                state=TaskState.TASK_STATE_COMPLETED, text="Done"))
        except Exception as e:
            logger.exception("Task %s failed", task.id)
            await event_queue.enqueue_event(new_text_status_update_event(
                task_id=task.id,
                context_id=task.context_id,
                state=TaskState.TASK_STATE_FAILED,
                text=str(e)))

    async def _process(self, text: str) -> str:
        if "fail" in text.lower():
            raise RuntimeError("upstream weather service unavailable")
        return await self._get_weather(text)


def build_card(base_url: str) -> AgentCard:
    return json_format.ParseDict({
        "name": "WeatherAgent",
        "description": "Returns a short forecast for a location",
        "version": "1.0.0",
        "supportedInterfaces": [{"url": f"{base_url}/a2a/v1", "protocolBinding": "JSONRPC", "protocolVersion": "1.0"}],
        "capabilities": {"streaming": True},
        "securitySchemes": {"bearer": {"httpAuthSecurityScheme": {"scheme": "bearer", "bearerFormat": "JWT"}}},
        "securityRequirements": [{"schemes": {"bearer": {"list": []}}}],
        "defaultInputModes": ["text/plain"],
        "defaultOutputModes": ["text/plain"],
        "skills": [{"id": "forecast", "name": "Forecast", "description": "Forecast for a named location", "tags": ["weather"]}],
    }, AgentCard())


def build_routes(base_url: str, executor: AgentExecutor) -> list:
    """Agent Card route plus the JSON-RPC binding, ready to mount on a Starlette app."""
    card = build_card(base_url)
    handler = DefaultRequestHandler(agent_executor=executor, task_store=InMemoryTaskStore(), agent_card=card)
    return [*create_agent_card_routes(card), *create_jsonrpc_routes(handler, rpc_url="/a2a/v1")]


def build_app(base_url: str, executor: AgentExecutor) -> Starlette:
    return Starlette(routes=build_routes(base_url, executor))


if __name__ == "__main__":
    port = int(sys.argv[1]) if len(sys.argv) > 1 else 8765
    logging.basicConfig(level=logging.ERROR)
    uvicorn.run(build_app(f"http://127.0.0.1:{port}", ProductionExecutor()),
                host="127.0.0.1", port=port, log_level="warning")

### Starting the agent

The server runs the production executor. This cell starts it in the background and waits for its port.

In [ ]:
WEATHER_PORT = 8765
WEATHER_BASE = f"http://127.0.0.1:{WEATHER_PORT}"

weather_process = subprocess.Popen(
    [PYTHON, "weather_agent.py", str(WEATHER_PORT)],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
wait_for_port(WEATHER_PORT)

card = requests.get(f"{WEATHER_BASE}/.well-known/agent-card.json", timeout=10).json()
print("=== WeatherAgent running ===")
print(f"pid {weather_process.pid}, interface {card['supportedInterfaces'][0]['url']}")
print(f"security schemes: {list(card['securitySchemes'])}, skills: {[s['id'] for s in card['skills']]}")

### Probing the binding

Before any SDK client touches the server, two raw JSON-RPC calls show what the executors put on the wire: a normal request that completes with a forecast artifact, and a request containing the word fail, which the production executor turns into a failed task whose status message carries the exception text.

In [ ]:
def send_raw(text: str) -> dict:
    """One SendMessage call over the JSON-RPC binding, no SDK involved."""
    payload = {
        "jsonrpc": "2.0", "id": 1, "method": "SendMessage",
        "params": {"message": {"messageId": str(uuid.uuid4()), "role": "ROLE_USER",
                               "parts": [{"text": text}]}},
    }
    response = requests.post(f"{WEATHER_BASE}/a2a/v1", json=payload,
                             headers={"A2A-Version": "1.0"}, timeout=30)
    response.raise_for_status()
    return response.json()["result"]["task"]


task = send_raw("Paris")
print("=== Success path ===")
print(f"state:    {task['status']['state']}")
print(f"artifact: {task['artifacts'][0]['name']} -> {task['artifacts'][0]['parts'][0]['text']}")

task = send_raw("Paris, please fail")
print("\n=== Failure path ===")
print(f"state:    {task['status']['state']}")
print(f"message:  {task['status']['message']['parts'][0]['text']}")

## Part 2: Integrating A2A with LangGraph

### Exposing a graph

LangChain's Agent Server exposes every deployed assistant at `/a2a/{assistant_id}` and generates its Agent Card. That is a deployment rather than a notebook cell, so the only code to show is the chapter's requirement on the graph itself: its state must carry a `messages` key, because the server maps incoming A2A text and file parts onto it.

In [ ]:
from typing import TypedDict

class AgentState(TypedDict):
    # A2A maps incoming text parts to this key, so it is required
    messages: list

print(f"AgentState keys: {list(AgentState.__annotations__)}")

### Consuming an A2A agent as a tool

The chapter's listing wraps a remote A2A agent in a LangChain `@tool`. The SDK's `create_client` fetches the Agent Card and picks a transport, `send_message` streams the responses, and the tool returns the text of the first artifact. The cell calls the tool directly first, then hands it to a `create_agent()` agent so the model decides on its own when to reach the weather agent.

In [ ]:
from langchain_core.tools import tool
from a2a.client import create_client
from a2a.helpers import new_text_message
from a2a.types import SendMessageRequest, Role

@tool
async def call_weather_agent(location: str) -> str:
    """Get a forecast from an external A2A weather agent."""
    client = await create_client(WEATHER_BASE)
    request = SendMessageRequest(message=new_text_message(
                                         location,
                                         role=Role.ROLE_USER))

    async for response in client.send_message(request):
        if response.HasField("artifact_update"):
            return response.artifact_update.artifact.parts[0].text

    return "Unavailable"


# --- Run it ---
print("=== The tool on its own ===")
print(await call_weather_agent.ainvoke({"location": "Berlin"}))

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL, reasoning_effort="none")
agent = create_agent(llm, [call_weather_agent])

result = await agent.ainvoke({
    "messages": [{"role": "user", "content": "I am flying to Lisbon tomorrow. Check the forecast with the weather agent and tell me whether to pack a jacket."}]
})

print("=== Conversation ===")
for message in result["messages"]:
    kind = type(message).__name__
    if getattr(message, "tool_calls", None):
        for call in message.tool_calls:
            print(f"{kind:13s} -> {call['name']}({call['args']})")
    elif message.content:
        text = message.content if isinstance(message.content, str) else json.dumps(message.content)
        print(f"{kind:13s} {text[:200]}")

## Part 3: CrewAI, Google ADK, and Strands Agents

These three frameworks wrap A2A behind their own configuration, and the chapter's listings for them appear below as reference cells rather than executable ones. The reason is a dependency conflict, not a gap in the code: `crewai[a2a]` and `strands-agents[a2a]` pin `a2a-sdk` to the 0.3 line, while everything else in this notebook runs on the 1.1.2 SDK that the chapter's pure Python examples target. Google ADK accepts either line but is not part of this repository's environment. To run these cells, create a separate environment for the framework in question and install it with its `a2a` extra.

### CrewAI delegation through an agent-level configuration

```python
from crewai import Agent, Task, Crew
from crewai.a2a import A2AClientConfig

# Delegation is configured on the agent via the `a2a` parameter
travel_planner = Agent(
    role="Travel Planner",
    goal="Plan comprehensive travel itineraries",
    backstory="Builds trips around real conditions",
    a2a=A2AClientConfig(
        endpoint="https://api.weather-services.com/.well-known/agent-card.json",
        timeout=120,
    ),
)

crew = Crew(agents=[travel_planner],
            tasks=[Task(
                description="Plan a 5-day Paris trip, accounting for the weather",
                agent=travel_planner)])

result = crew.kickoff()
```

### CrewAI collaborating with an agent built on another framework

```python
from crewai import Agent, Task, Crew
from crewai.a2a import A2AClientConfig

# The endpoint happens to be a LangGraph agent; A2A hides that detail
data_collector = Agent(
    role="Data Collector",
    goal="Gather sales data and delegate the analysis",
    backstory="Coordinates specialists across frameworks",
    a2a=A2AClientConfig(
        endpoint="https://langgraph-server.com/.well-known/agent-card.json"),
)

report_writer = Agent(role="Report Writer",
                      goal="Summarize the analysis",
                      backstory="Turns analysis into a clear report")

crew = Crew(agents=[data_collector, report_writer],
            tasks=[Task(
                description="Collect sales data, delegate analysis, then summarize",
                agent=data_collector)])

result = crew.kickoff()
```

### Google ADK

```python
from google.adk.agents import Agent
from google.adk.a2a.utils.agent_to_a2a import to_a2a

def analyze_balance_sheet(current_assets: float,
                          current_liabilities: float) -> dict:
    """Compute the current ratio from a balance sheet."""
    return {"current_ratio": round(current_assets / current_liabilities, 2)}

financial_analyst = Agent(
    name="financial_analyst",
    model="gemini-flash-latest",
    description="Analyzes financial statements",
    instruction="Analyze balance sheets and report key ratios.",
    tools=[analyze_balance_sheet],
)

# Wrap as an A2A server app; the Agent Card is generated automatically
a2a_app = to_a2a(financial_analyst, port=8080)

# Serve with: uvicorn financial_analyst:a2a_app --port 8080
```

### Strands Agents

```python
from strands import Agent, tool
from strands.multiagent.a2a import A2AServer


@tool
def check_system_health(system_name: str) -> dict:
    """Check the health of a named system."""
    return {"system": system_name, "status": "healthy", "cpu": "45%"}


def create_agent(context_id: str) -> Agent:
    return Agent(
        name="IT Diagnostic Agent",
        description="Diagnoses IT infrastructure issues",
        tools=[check_system_health],
    )


# Serve a fresh agent per conversation context over A2A
A2AServer(agent_factory=create_agent, port=9000).serve()
```

## Part 4: Multi-Agent Workflow Patterns

### The specialist agent

The patterns delegate to a data collector, an analyzer, a report writer, and later to three hospital agents. Rather than start six servers, the file below starts one scripted agent that answers by the first words of the instruction, fails on request, and denies pre-authorization for patient ids ending in 9. The answers are fixed so the workflow outputs are readable; the delegation code does not know or care that one process is behind every URL.

In [ ]:
%%writefile specialist_agent.py
"""One A2A server that plays every specialist in the workflow patterns, with scripted answers."""
import sys

import uvicorn
from google.protobuf import json_format
from starlette.applications import Starlette

from a2a.helpers import (new_task_from_user_message, new_text_artifact_update_event,
                         new_text_status_update_event)
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import AgentCard, TaskState

PORT = int(sys.argv[1]) if len(sys.argv) > 1 else 8766

# Each entry: (keyword the instruction starts with, reply). Anything else is echoed back.
SCRIPT = [
    ("collect data", "Q3 sales data: 1,240 units sold, revenue 3.1M, returns 2.4 percent, top region EMEA"),
    ("analyze", "Analysis: revenue up 12 percent quarter on quarter while units stayed flat, so average price rose; returns are within the 3 percent target"),
    ("write a report", "Q3 report: revenue grew 12 percent on flat volume through pricing, returns held at 2.4 percent, EMEA led; recommendation is to keep pricing and watch volume"),
    ("pre-authorize", None),  # decided per patient below
    ("schedule", "Cardiology appointment booked for 2026-10-03 at 09:30 with Dr. Rao"),
    ("notify", "Patient notified by SMS and portal message"),
    ("get paris weather", "Paris: 19 C, light rain in the afternoon"),
    ("find paris hotels", "Three hotels near the Marais under 220 EUR with availability"),
]


def respond(instruction: str) -> str:
    text = instruction.strip()
    lowered = text.lower()
    if "fail" in lowered:
        raise RuntimeError("specialist unavailable")
    for keyword, reply in SCRIPT:
        if lowered.startswith(keyword):
            if keyword == "pre-authorize":
                # Patient ids ending in 9 are denied so the gate in the referral workflow can be seen
                return "denied: plan does not cover out-of-network cardiology" if text.rstrip().endswith("9") \
                    else "approved: policy covers cardiology referral, authorization AUTH-88213"
            return reply
    return f"Handled: {text}"


class SpecialistExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        task = context.current_task or new_task_from_user_message(context.message)
        await event_queue.enqueue_event(task)
        try:
            reply = respond(context.get_user_input())
        except Exception as exc:
            await event_queue.enqueue_event(new_text_status_update_event(
                task.id, task.context_id, TaskState.TASK_STATE_FAILED, str(exc)))
            return
        await event_queue.enqueue_event(new_text_artifact_update_event(task.id, task.context_id, "result", reply))
        await event_queue.enqueue_event(new_text_status_update_event(
            task.id, task.context_id, TaskState.TASK_STATE_COMPLETED, "Done"))

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        await event_queue.enqueue_event(new_text_status_update_event(
            context.task_id, context.context_id, TaskState.TASK_STATE_CANCELED, "Canceled"))


card = json_format.ParseDict({
    "name": "SpecialistAgent",
    "description": "Plays the data collector, analyzer, report writer, and hospital agents with scripted answers",
    "version": "1.0.0",
    "supportedInterfaces": [{"url": f"http://127.0.0.1:{PORT}/a2a/v1", "protocolBinding": "JSONRPC", "protocolVersion": "1.0"}],
    "capabilities": {"streaming": True},
    "defaultInputModes": ["text/plain"],
    "defaultOutputModes": ["text/plain"],
    "skills": [{"id": "specialist", "name": "Specialist", "description": "Scripted specialist for workflow demos", "tags": ["demo"]}],
}, AgentCard())
handler = DefaultRequestHandler(agent_executor=SpecialistExecutor(), task_store=InMemoryTaskStore(), agent_card=card)
app = Starlette(routes=[*create_agent_card_routes(card), *create_jsonrpc_routes(handler, rpc_url="/a2a/v1")])

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning")

In [ ]:
SPECIALIST_PORT = 8766
SPECIALIST_BASE = f"http://127.0.0.1:{SPECIALIST_PORT}"

specialist_process = subprocess.Popen(
    [PYTHON, "specialist_agent.py", str(SPECIALIST_PORT)],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
wait_for_port(SPECIALIST_PORT)
print(f"=== SpecialistAgent running on {SPECIALIST_BASE} (pid {specialist_process.pid}) ===")

### The delegate helper

Every pattern goes through one helper. It wraps the SDK client, sends one instruction, and returns the text of the first artifact, whether that arrives as a streaming artifact event or inside a completed task. A task that ends in the failed state becomes an exception carrying the agent's message.

In [ ]:
from a2a.types import TaskState


async def delegate(agent_url: str,
                   instruction: str) -> str:
    """Send one instruction to a remote A2A agent and return its text result."""
    client = await create_client(agent_url)
    request = SendMessageRequest(message=new_text_message(
        instruction, role=Role.ROLE_USER))

    async for response in client.send_message(request):
        if response.HasField("artifact_update"):
            return response.artifact_update.artifact.parts[0].text
        if response.HasField("task") and response.task.artifacts:
            return response.task.artifacts[0].parts[0].text
        if response.HasField("status_update") and \
                response.status_update.status.state == TaskState.TASK_STATE_FAILED:
            raise RuntimeError(response.status_update.status.message.parts[0].text)

    return ""


# --- Run it ---
print(await delegate(SPECIALIST_BASE, "Collect data: Q3 sales"))
try:
    await delegate(SPECIALIST_BASE, "please fail")
except RuntimeError as exc:
    print(f"failed task surfaced as an exception: {exc}")

### Sequential delegation

The orchestrator calls the collector, feeds its output to the analyzer, and feeds the analysis to the writer. The dependency forces the order.

In [ ]:
AGENTS = {
    "collector": SPECIALIST_BASE,
    "analyzer":  SPECIALIST_BASE,
    "writer":    SPECIALIST_BASE,
}

async def run_workflow(request: str) -> str:
    raw_data = await delegate(AGENTS["collector"],
                             f"Collect data: {request}")
    analysis = await delegate(AGENTS["analyzer"],
                             f"Analyze: {raw_data}")
    report   = await delegate(AGENTS["writer"],
                             f"Write a report from: {analysis}")
    return  report


# --- Run it ---
started = time.time()
print(await run_workflow("Q3 sales"))
print(f"\nthree sequential delegations took {time.time() - started:.2f}s")

### Parallel execution

Independent tasks start at once with `asyncio.gather`. With `return_exceptions=True`, the delegation that fails comes back as an error object in the results list instead of cancelling the batch, which is why the helper raises on a failed task.

In [ ]:
async def run_parallel(tasks: list[tuple[str, str]]) -> list[str]:
    results = await  asyncio.gather(
        *(delegate(url, instruction) for url, instruction in tasks),
        return_exceptions=True,
    )
    return  [str(r) for r in results]


# --- Run it ---
started = time.time()
results = await run_parallel([
    (SPECIALIST_BASE, "Get Paris weather"),
    (SPECIALIST_BASE, "Find Paris hotels"),
    (SPECIALIST_BASE, "Book the flight, and fail if the fare is missing"),
])
for line in results:
    print(f"- {line}")
print(f"\nthree parallel delegations took {time.time() - started:.2f}s")

### Human-in-the-loop

The chapter's orchestrator asks whether a request needs approval, notifies a person, and polls for the decision under a one-hour timeout. The four functions it calls are the integration points with your approval system; the stand-ins below approve any request on the second check, so the cell finishes after one polling interval. In a full A2A implementation the agent side would park its task in `TASK_STATE_AUTH_REQUIRED` with the question in the status message, as the specification's in-task authorization section describes, and the client would continue the task once the person decided.

In [ ]:
APPROVALS: dict[str, dict] = {}


def needs_approval(request: str) -> bool:
    return any(word in request.lower() for word in ("delete", "transfer", "refund"))


async def request_approval(request: str) -> str:
    approval_id = str(uuid.uuid4())[:8]
    APPROVALS[approval_id] = {"request": request, "checks": 0}
    print(f"  [approval requested] {approval_id}: {request}")
    return approval_id


async def check_approval(approval_id: str) -> str:
    record = APPROVALS[approval_id]
    record["checks"] += 1
    return "approved" if record["checks"] >= 2 else "pending"


async def act_on(request: str) -> str:
    return await delegate(SPECIALIST_BASE, request)


async def execute_with_approval(request: str) -> str:
    if not needs_approval(request):
        return  await  act_on(request)

    # notify a human, e.g. via webhook
    approval_id = await  request_approval(request)

    # stop waiting after an hour
    async with asyncio.timeout(3600):
        while True:
            status = await  check_approval(approval_id)
            if status == "approved":
                return  await act_on(request)
            if status == "rejected":
                return  "Request rejected"
            await  asyncio.sleep(5)


# --- Run it ---
print("=== No approval needed ===")
print(await execute_with_approval("Get Paris weather"))
print("\n=== Approval needed (approved on the second check, about 5 seconds) ===")
print(await execute_with_approval("Refund order 4471 in full"))

## Part 5: The Healthcare Referral Workflow

The referral workflow is the sequential pattern with a gate. Insurance pre-authorization runs first; if coverage is denied the orchestrator stops and returns the reason, and only on approval does it schedule the appointment and notify the patient. The specialist agent approves patient P-1042 and denies P-2039.

In [ ]:
HOSPITAL_AGENTS = {
    "insurance":    SPECIALIST_BASE,
    "scheduling":   SPECIALIST_BASE,
    "notification": SPECIALIST_BASE,
}

async def process_referral(referral: dict) -> dict:
    patient = referral["patient_id"]

    # Pre-authorization gates the rest of the workflow
    auth = await  delegate(HOSPITAL_AGENTS["insurance"],
                          f"Pre-authorize referral for patient {patient}")
    if "denied" in auth.lower():
        return  {"status": "denied", "reason": auth}

    appointment = await  delegate(HOSPITAL_AGENTS["scheduling"],
               f"Schedule {referral['specialty']} for patient {patient}")

    await  delegate(HOSPITAL_AGENTS["notification"],
                   f"Notify patient {patient}: {appointment}")

    return  {"status": "completed", "appointment": appointment}


# --- Run it ---
for patient_id in ("P-1042", "P-2039"):
    outcome = await process_referral({"patient_id": patient_id, "specialty": "cardiology"})
    print(f"{patient_id}: {json.dumps(outcome)}")

## Part 6: Security

### TLS 1.3 enforcement

The SDK does not handle TLS; the server that hosts the app does. uvicorn takes the certificate and key through `ssl_certfile` and `ssl_keyfile`, and it calls `ssl_context_factory` with its config and a default factory that loads them. The chapter's function calls that default factory and raises the minimum version to TLS 1.3, which turns the specification's recommendation into enforcement.

The cell generates a self-signed certificate with `openssl`, serves the weather app over HTTPS in a background thread (the chapter's `uvicorn.run()` call is the blocking form of the same thing), and probes it twice: a client that speaks TLS 1.3 connects, and a client capped at TLS 1.2 is refused during the handshake.

In [ ]:
import ssl
import uvicorn
from weather_agent import WeatherAgentExecutor, build_app, build_routes

subprocess.run(
    ["openssl", "req", "-x509", "-newkey", "rsa:2048", "-nodes", "-days", "1",
     "-keyout", "server.key", "-out", "server.crt", "-subj", "/CN=localhost"],
    check=True, capture_output=True,
)


def tls_context(config, default_factory) -> ssl.SSLContext:
    ctx = default_factory()  # loads the cert and key passed to uvicorn.run
    ctx.minimum_version = ssl.TLSVersion.TLSv1_3  # reject anything below TLS 1.3
    return ctx


# --- Run it ---
TLS_PORT = 8443
tls_server = uvicorn.Server(uvicorn.Config(
    build_app(f"https://127.0.0.1:{TLS_PORT}", WeatherAgentExecutor()),
    host="127.0.0.1", port=TLS_PORT, log_level="warning",
    ssl_certfile="server.crt", ssl_keyfile="server.key",
    ssl_context_factory=tls_context,
))
threading.Thread(target=tls_server.run, daemon=True).start()
wait_for_port(TLS_PORT)

for label, max_version in (("TLS 1.3 client", ssl.TLSVersion.TLSv1_3),
                           ("TLS 1.2 client", ssl.TLSVersion.TLSv1_2)):
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE   # self-signed certificate; production clients verify against a CA
    ctx.maximum_version = max_version
    try:
        with socket.create_connection(("127.0.0.1", TLS_PORT)) as sock, \
                ctx.wrap_socket(sock, server_hostname="localhost") as tls:
            print(f"{label}: handshake ok, negotiated {tls.version()}")
    except ssl.SSLError as exc:
        print(f"{label}: refused ({exc.reason})")

# The card is served over HTTPS like any other route
print(httpx.get(f"https://127.0.0.1:{TLS_PORT}/.well-known/agent-card.json", verify=False).json()["name"])
tls_server.should_exit = True

### Bearer token authentication

The chapter's middleware reads the bearer token from the Authorization header and rejects the request with a 401 when it is missing or invalid. The real check lives in `validate_token`, which the next cell defines. This cell attaches the middleware to the weather agent's own routes, so the A2A binding sits behind authentication.

In [ ]:
from starlette.applications import Starlette
from starlette.middleware import Middleware
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.responses import JSONResponse

class BearerAuthMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request, call_next):
        header = request.headers.get("Authorization", "")
        token = header[7:] if header.startswith("Bearer ") else None
        if not token or not await validate_token(token):
            return JSONResponse({"error": "unauthorized"}, status_code=401)
        return await call_next(request)


# Attach to the Starlette app built in the server section
routes = build_routes("http://testserver", WeatherAgentExecutor())
app = Starlette(routes=routes,
                middleware=[Middleware(BearerAuthMiddleware)])
print("bearer-protected app built with", len(routes), "A2A routes")

### OAuth 2.0 access tokens and API keys

An OAuth 2.0 access token is a bearer token, so it flows through the same middleware; only `validate_token` changes, verifying the token's signature and checking for the required scope. API keys get their own middleware that checks a custom header. The cell generates an RSA key pair to stand in for the authorization server, signs two tokens with it, and tests both apps with an in-process HTTP client. The last request is a real `SendMessage` through the authenticated app.

In [ ]:
import jwt  # PyJWT, installed via pyjwt[crypto]
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import rsa

# Stand-in for the authorization server's signing key (in production, fetch the public key from its JWKS endpoint)
_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
IDP_PRIVATE_KEY = _key.private_bytes(serialization.Encoding.PEM, serialization.PrivateFormat.PKCS8,
                                     serialization.NoEncryption())
IDP_PUBLIC_KEY = _key.public_key().public_bytes(serialization.Encoding.PEM,
                                                serialization.PublicFormat.SubjectPublicKeyInfo)
VALID_API_KEYS = {"key-" + uuid.uuid4().hex[:12]}


# API key: validate a key from a custom header
class APIKeyMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request, call_next):
        if request.headers.get("X-API-Key") not in VALID_API_KEYS:
            return JSONResponse({"error": "unauthorized"},
                                status_code=401)
        return await call_next(request)


# OAuth 2.0: an access token is a bearer token, so reuse BearerAuthMiddleware
# with a validator that verifies the token's signature and scope
async def validate_token(token: str) -> bool:
    try:
        claims = jwt.decode(token, key=IDP_PUBLIC_KEY,
                            algorithms=["RS256"], audience="agent-api")
    except jwt.PyJWTError:
        return False

    return "agent.execute" in claims.get("scope", "").split()


# --- Run it ---
good_token = jwt.encode({"aud": "agent-api", "scope": "agent.read agent.execute"}, IDP_PRIVATE_KEY, algorithm="RS256")
read_only = jwt.encode({"aud": "agent-api", "scope": "agent.read"}, IDP_PRIVATE_KEY, algorithm="RS256")

async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url="http://testserver") as http:
    print("=== Bearer / OAuth 2.0 ===")
    for label, headers in [("no header", {}),
                           ("garbage token", {"Authorization": "Bearer nope"}),
                           ("token without scope", {"Authorization": f"Bearer {read_only}"}),
                           ("token with scope", {"Authorization": f"Bearer {good_token}"})]:
        status = (await http.get("/.well-known/agent-card.json", headers=headers)).status_code
        print(f"{label:20s} -> {status}")

    payload = {"jsonrpc": "2.0", "id": 1, "method": "SendMessage",
               "params": {"message": {"messageId": str(uuid.uuid4()), "role": "ROLE_USER",
                                      "parts": [{"text": "Madrid"}]}}}
    response = await http.post("/a2a/v1", json=payload,
                               headers={"Authorization": f"Bearer {good_token}", "A2A-Version": "1.0"})
    task = response.json()["result"]["task"]
    print(f"authenticated SendMessage -> {task['status']['state']}: {task['artifacts'][0]['parts'][0]['text']}")

key_app = Starlette(routes=build_routes("http://testserver", WeatherAgentExecutor()),
                    middleware=[Middleware(APIKeyMiddleware)])
async with httpx.AsyncClient(transport=httpx.ASGITransport(app=key_app), base_url="http://testserver") as http:
    print("\n=== API key ===")
    print(f"missing key -> {(await http.get('/.well-known/agent-card.json')).status_code}")
    print(f"valid key   -> {(await http.get('/.well-known/agent-card.json', headers={'X-API-Key': next(iter(VALID_API_KEYS))})).status_code}")

### Role-based authorization

Authentication says who is calling; authorization decides what the caller may do. The chapter's authorizer maps each 1.0 method name to an action and checks the caller's role. `SendMessage`, `SendStreamingMessage`, and `CancelTask` are execute operations; everything else is a read. The role comes from the verified token's claims, never from the request body, and an unknown role gets no permissions at all.

In [ ]:
EXECUTE_METHODS = {"SendMessage", "SendStreamingMessage", "CancelTask"}


class RBACAuthorizer:
    """Role-based access control for A2A agents."""

    def __init__(self):
        self.role_permissions = {
            "admin": {"*"},
            "analyst": {"read", "execute"},
            "viewer": {"read"},
        }

    def authorize(self, role: str, method: str) -> bool:
        """`role` comes from the verified token, never from the request body."""
        action = "execute" if method in EXECUTE_METHODS else "read"
        permissions = self.role_permissions.get(role, set())
        return "*" in permissions or action in permissions


# --- Run it ---
authorizer = RBACAuthorizer()
print(f"{'role':10s} {'method':24s} allowed")
for role, method in [("viewer", "GetTask"), ("viewer", "SendMessage"), ("analyst", "SendStreamingMessage"),
                     ("analyst", "CancelTask"), ("admin", "DeleteTaskPushNotificationConfig"), ("intern", "GetTask")]:
    print(f"{role:10s} {method:24s} {authorizer.authorize(role, method)}")

## Part 7: A Model-Driven Orchestrator over A2A

The sequential pattern in Part 4 hard-codes the order of the three delegations. Here a model decides. Each specialist becomes a LangChain tool that calls `delegate`, and a `create_agent()` agent on `gpt-5.6-luna` is asked for the Q3 report. The model has to work out that it needs the raw data before the analysis and the analysis before the report, and every one of its calls travels over A2A to the specialist agent.

In [ ]:
@tool
async def collect_sales_data(request: str) -> str:
    """Ask the data collector agent for raw sales data matching the request."""
    return await delegate(AGENTS["collector"], f"Collect data: {request}")


@tool
async def analyze_sales_data(raw_data: str) -> str:
    """Ask the analyzer agent to analyze raw sales data and return its findings."""
    return await delegate(AGENTS["analyzer"], f"Analyze: {raw_data}")


@tool
async def write_sales_report(analysis: str) -> str:
    """Ask the report writer agent to turn an analysis into a finished report."""
    return await delegate(AGENTS["writer"], f"Write a report from: {analysis}")


orchestrator = create_agent(
    ChatOpenAI(model=MODEL, reasoning_effort="none"),
    [collect_sales_data, analyze_sales_data, write_sales_report],
    system_prompt="You coordinate specialist agents. Use the tools to gather data, analyze it, and produce the report, then return the report text.",
)

result = await orchestrator.ainvoke({
    "messages": [{"role": "user", "content": "Produce the Q3 sales report."}]
})

print("=== Delegations chosen by the model ===")
for message in result["messages"]:
    if getattr(message, "tool_calls", None):
        for call in message.tool_calls:
            print(f"-> {call['name']}")
print("\n=== Final answer ===")
final = result["messages"][-1].content
print(final if isinstance(final, str) else json.dumps(final))

### Stopping the agents

The HTTPS server was told to exit in Part 6. The two agent processes are ours to stop.

In [ ]:
for name, proc in [("WeatherAgent", weather_process), ("SpecialistAgent", specialist_process)]:
    proc.terminate()
    proc.wait(timeout=5)
    print(f"{name} stopped with return code {proc.returncode}")

## Summary

In this notebook, we implemented:

1. **An SDK Server**: The chapter's two executors behind the SDK's request handler and route factories, probed with raw JSON-RPC on both the success and failure paths
2. **LangGraph Integration**: The `messages` state requirement for exposing a graph, and an A2A agent consumed as a tool by a `create_agent()` agent
3. **Framework Reference Cells**: The CrewAI, ADK, and Strands listings, with the SDK version conflict that keeps them out of this environment
4. **Workflow Patterns**: The `delegate` helper, sequential and parallel delegation against a scripted specialist agent, and human-in-the-loop approval with stand-in hooks
5. **The Referral Workflow**: Pre-authorization as a gate, with one patient approved and one denied
6. **Security**: TLS 1.3 enforced through uvicorn's context factory, bearer and API key middleware in front of the A2A routes, a JWT validator with scope checks, and role-based authorization on 1.0 method names
7. **A Model-Driven Orchestrator**: Three A2A specialists as tools, sequenced by the model rather than by code

### Key Takeaways:

- An executor's contract is the event queue: Task first, then updates, ending in a terminal state, with failures reported as `TASK_STATE_FAILED`
- Consuming an A2A agent from a framework is a tool wrapper around the SDK client; exposing one is a deployment concern the frameworks handle differently
- The workflow patterns differ only in how they schedule the same `delegate` call
- Security is layered: encrypted transport, then authentication middleware, then authorization on the authenticated identity
- The 0.3 and 1.x SDK lines cannot share an environment, which decides where each framework's code can run

### Next Steps:

- Replace the scripted specialist with real agents, one per URL, and let the orchestrator discover them through their Agent Cards
- Move the approval hooks onto the agent side with `TASK_STATE_AUTH_REQUIRED` so the client sees the pause in the task itself
- Move on to Chapter 9, which deploys distributed agent systems in production